# 24 -

## Set Up

### Libraries

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
#===
import sys
from pathlib import Path

### Paths

In [2]:
project_root = Path().resolve().parent.parent
sys.path.append(str(project_root))
project_root

PosixPath('/Users/alejandrofp/Desktop/Projects/03_Flagship_Portfolio/job-intelligence-engine')

In [3]:
from src.job_intel.positioning import run_positioning
from src.job_intel.features.artefacts_ch3 import load_ch3_artefacts

In [4]:
profile, candidates_df, gap_df = run_positioning(
    skill_text=(
        "Python, SQL, machine learning, statistics, data analysis, "
        "scikit-learn, pandas, experimentation"
    ),
    current_state="CA",
    job_title_family="data_scientist",
    salary_target=150000,
    return_top_n_jobs=50,
)

_, skill_prob_matrix = load_ch3_artefacts()

# 1) user missing-skill indicator (shape: 27,)
user_vec = profile["derived"]["skill_vector"].iloc[0].to_numpy()  # 0/1
missing_vec = 1 - user_vec                                       # 1 where user lacks skill

# 2) probability columns aligned to user skill order
skill_cols = profile["derived"]["skill_vector"].columns.tolist()
prob_cols = [f"{s}_prob" for s in skill_cols]

# 3) align matrix rows to candidates (job_id must exist in both)
probs = (
    skill_prob_matrix.loc[
        skill_prob_matrix["job_id"].isin(candidates_df["job_id"]),
        ["job_id"] + prob_cols,
    ]
    .set_index("job_id")
    .loc[candidates_df["job_id"]]  # re-order to match candidates_df
)

# 4) expected missing per job (dot product)
expected_missing = probs.to_numpy() @ missing_vec

# 5) attach + normalize to [0,1]
candidates_df = candidates_df.copy()
candidates_df["expected_missing"] = expected_missing
candidates_df["expected_missing_norm"] = candidates_df["expected_missing"] / len(skill_cols)

# 6) salary percentile barrier in [0,1]
candidates_df["salary_pct"] = candidates_df["sal_mean"].rank(pct=True, method="average")

# 7) competitiveness index in [0,1]
candidates_df["competitiveness_index"] = (
    0.5 * candidates_df["expected_missing_norm"]
    + 0.5 * candidates_df["salary_pct"]
)


In [6]:
candidates_df[["sal_mean", "salary_pct"]].describe()
candidates_df.sort_values("salary_pct", ascending=False)[["title_rich", "sal_mean", "salary_pct"]].head(5)


,title_rich,sal_mean,salary_pct
2993,general_data_data_scientist,225000.0,0.97
2998,general_data_data_scientist,225000.0,0.97
2991,general_data_data_scientist,225000.0,0.97
2994,ML_AI_data_scientist,225000.0,0.97
2963,ML_AI_data_scientist,193500.0,0.92


## Skill Rarity Integration

In [7]:
global_skill_p_mean = skill_prob_matrix.drop('job_id', axis=1).mean(axis= 0).rename('global_average').reset_index()
eps = 0.00001
global_skill_p_mean['rarity_weights'] = 1 / (global_skill_p_mean['global_average'] + eps)

global_skill_p_mean['skills'] = profile["derived"]["skill_vector"].columns

global_skill_p_mean['weight_norm'] = global_skill_p_mean['rarity_weights'] / global_skill_p_mean['rarity_weights'].mean()

In [10]:
# --- Build global rarity weights (already mostly done) ---
global_skill_p_mean = (
    skill_prob_matrix
    .drop(columns=["job_id"])
    .mean(axis=0)
    .rename("global_average")
    .reset_index()
    .rename(columns={"index": "prob_col"})
)

eps = 1e-5
global_skill_p_mean["rarity_weight"] = 1 / (global_skill_p_mean["global_average"] + eps)

# Convert "skillname_prob" -> "skillname" (canonical key)
global_skill_p_mean["skill"] = global_skill_p_mean["prob_col"].str.replace("_prob", "", regex=False)

# Mean-normalise (stable scale)
global_skill_p_mean["weight_norm"] = global_skill_p_mean["rarity_weight"] / global_skill_p_mean["rarity_weight"].mean()

# --- Align weights to the user skill column order (critical) ---
skill_cols = profile["derived"]["skill_vector"].columns.tolist()

w_lookup = (
    global_skill_p_mean
    .set_index("skill")["weight_norm"]
    .reindex(skill_cols)
)

assert w_lookup.isna().sum() == 0, "Rarity weights missing for some skills (alignment failure)."

w_vec = w_lookup.to_numpy()  # length 27, aligned to skill_cols

# --- Apply rarity to missingness (vectorised) ---
prob_cols = [f"{s}_prob" for s in skill_cols]

probs = (
    skill_prob_matrix.loc[
        skill_prob_matrix["job_id"].isin(candidates_df["job_id"]),
        ["job_id"] + prob_cols,
    ]
    .set_index("job_id")
    .loc[candidates_df["job_id"]]
)

user_vec = profile["derived"]["skill_vector"].iloc[0].to_numpy()
missing_vec = 1 - user_vec

# rarity-weighted missing burden (sum across skills)
expected_missing_rarity = (probs.to_numpy() * w_vec) @ missing_vec

# normalise to ~0–1 scale (mean-normalised weights => divide by n_skills)
expected_missing_rarity_norm = expected_missing_rarity / len(skill_cols)

candidates_df = candidates_df.copy()
candidates_df["expected_missing_rarity"] = expected_missing_rarity
candidates_df["expected_missing_rarity_norm"] = expected_missing_rarity_norm.clip(0, 1)

# quick sanity
global_skill_p_mean.sort_values("weight_norm", ascending=False)[["skill","global_average","weight_norm"]]


,skill,global_average,weight_norm
2,core_programming__advanced,0.002108,11.289112
23,productivity_workflow__advanced,0.002775,8.584805
14,bi_viz__advanced,0.017947,1.331542
11,analytics_stats__advanced,0.019657,1.215762
20,db_storage__advanced,0.025986,0.919763
17,cloud__advanced,0.027831,0.858805
3,data_engineering_pipelines__basic,0.035023,0.682515
8,ml_ai__advanced,0.093998,0.254344
16,cloud__intermediate,0.104146,0.229562
1,core_programming__intermediate,0.113261,0.211089


In [11]:
candidates_df

,job_id,Job Description,Rating,Size,Founded,Industry,Sector,role_source,state,ownership_clean,...,skill_match_score,skill_match_norm,salary_score,suitability,expected_missing,expected_missing_norm,salary_pct,competitiveness_index,expected_missing_rarity,expected_missing_rarity_norm
3025,3025,Facebook's mission is to give people the power...,4.5,10000+ employees,2004.0,Internet,Information Technology,data_scientist,CA,public,...,0.831746,0.915873,1.000000,0.941111,3.699214,0.137008,0.84,0.488504,1.427915,0.052886
2870,2870,Entefy’s Senior Data Scientist is a highly vis...,4.3,1 to 50 employees,2012.0,Internet,Information Technology,data_scientist,CA,private,...,0.820711,0.910355,1.000000,0.937249,2.832693,0.104915,0.64,0.372457,0.430645,0.015950
3033,3033,Requisition ID: 255369\nWork Area: Software-Re...,4.6,10000+ employees,1972.0,Computer Hardware & Software,Information Technology,data_scientist,CA,public,...,0.794611,0.897305,1.000000,0.928114,1.682754,0.062324,0.84,0.451162,0.235272,0.008714
3130,3130,Facebook's mission is to give people the power...,4.5,10000+ employees,2004.0,Internet,Information Technology,data_scientist,CA,public,...,0.794061,0.897030,1.000000,0.927921,2.656831,0.098401,0.40,0.249201,1.022571,0.037873
382,382,About Opendoor:Are you intrigued by the though...,3.5,501 to 1000 employees,2014.0,Real Estate,Real Estate,data_scientist,CA,private,...,0.793145,0.896572,0.976667,0.920601,2.945821,0.109104,0.28,0.194552,0.283896,0.010515
3112,3112,Role: Senior Data Scientist\n\nLocation: Sunny...,NaN,Unknown,NaN,Unknown,Unknown,data_scientist,CA,unknown,...,0.742015,0.871007,1.000000,0.909705,2.427881,0.089922,0.46,0.274961,0.198034,0.007335
2873,2873,You must have experience in build and evaluate...,4.0,1 to 50 employees,NaN,Accounting,Accounting & Legal,data_scientist,CA,private,...,0.686077,0.843039,1.000000,0.890127,1.591933,0.058960,0.64,0.349480,0.148950,0.005517
2883,2883,Axiom Group has partnered with an elite group ...,4.7,51 to 200 employees,1987.0,Transportation Equipment Manufacturing,Manufacturing,data_scientist,CA,private,...,0.659743,0.829872,1.000000,0.880910,3.523000,0.130481,0.64,0.385241,0.542721,0.020101
2881,2881,Job Description\nTitle: Data Scientist\n\nLoca...,NaN,Unknown,NaN,Unknown,Unknown,data_scientist,CA,unknown,...,0.650377,0.825189,1.000000,0.877632,2.208478,0.081795,0.64,0.360898,0.287257,0.010639
3141,3141,The successful candidate will have experience ...,NaN,Unknown,NaN,Unknown,Unknown,data_scientist,CA,unknown,...,0.644752,0.822376,1.000000,0.875663,4.478726,0.165879,0.40,0.282939,0.567622,0.021023


## Sensitivity